In [0]:
from pyspark.sql.types import *
#Defined Schemas - per dataset

schemas = {
    "orders" : StructType([
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True), 
        StructField("order_status", StringType(), True),  
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True), 
        StructField("order_delivered_carrier_date", TimestampType(), True), 
        StructField("order_delivered_customer_date", TimestampType(), True), 
        StructField("order_estimated_delivery_date", TimestampType(), True)
    ]),
     "customers" : StructType([
        StructField("customer_id", StringType(), True), 
        StructField("customer_unique_id", StringType(), True), 
        StructField("customer_zip_code_prefix", IntegerType(), True), 
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True)
    ]),
     "products" : StructType([
        StructField("product_id", StringType(), True), 
        StructField("product_category_name", StringType(), True), 
        StructField("product_name_lenght", IntegerType(), True), 
        StructField("product_description_lenght", IntegerType(), True), 
        StructField("product_photos_qty", IntegerType(), True), 
        StructField("product_weight_g", IntegerType(), True), 
        StructField("product_lenghth_cm",IntegerType(),True),
        StructField("product_height_cm",IntegerType(),True),
        StructField("product_width_cm",IntegerType(),True)
    ]),
     "order_items" : StructType([
        StructField("order_id",StringType(),True),
        StructField("order_item_id",IntegerType(),True),
        StructField("product_id",StringType(),True),
        StructField("seller_id",StringType(),True),
        StructField("shipping_limit_date",TimestampType(),True),
        StructField("price",DoubleType(),True),
        StructField("freight_value",DoubleType(),True)
     ]),
     "order_payments" : StructType([
       StructField("order_id",StringType(),True),
       StructField("payment_sequential",IntegerType(),True),
       StructField("payment_type",StringType(),True),
       StructField("payment_installments",IntegerType(),True),
       StructField("payment_value",DoubleType(),True) 
     ])        
}


Bronze_Path = "/Volumes/main/ecommerce/lakehouse_vol/Bronze/"
#Managed_Path = "/Volumes/main/ecommerce/"

for dataset, schema1 in schemas.items():
#Read operation Bronze --> silver    
   df = spark.read.schema(schema1).parquet(Bronze_Path + dataset)

   #Remove Null
   df = df.dropna(subset=[schema1.fields[0].name])
   #Remove duplications - using primary key
   df = df.dropDuplicates([schema1.fields[0].name])

   #Write operation Silver --> Managed Delta Table
   df.write.mode("overwrite").format("delta").saveAsTable(f"main.ecommerce.{dataset}")
   #Optional: Write operation Bronze --> Siver
   df.repartition(1).write.format("delta").mode("overwrite").save(f"/Volumes/main/ecommerce/lakehouse_vol/Silver/{dataset}")
